# Florence-2 Fine-tuning — Roboflow Dataset

Notebook upravený pre server s **32-core CPU, 128GB RAM, NVIDIA L40S 48GB**.
Pripravené špeciálne pre formát datasetu exportovaného z Roboflow pre model Florence-2.

## 1. Inštalácia závislostí a importy

In [ ]:
!pip install -q -U datasets bitsandbytes peft transformers accelerate Pillow
import torch
import json
import os
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, PeftModel

## 2. Konfigurácia (Uprav cesty k datasetu)

In [ ]:
# ==================== KONFIGURÁCIA ====================
DATASET_PATH  = "PERSON-WEAPON-FINISHED-1/dataset/_annotations.train.jsonl"
IMAGE_FOLDER  = "PERSON-WEAPON-FINISHED-1/dataset/"

# Môžeš použiť microsoft/Florence-2-large alebo -base
MODEL_ID      = "microsoft/Florence-2-large"
OUTPUT_DIR    = "florence2_weapon_finetune"
HF_REPO_ID    = None

# Tréningové hyperparametre (rovnaké overené hodnoty z PaliGemmy)
NUM_EPOCHS    = 5
BATCH_SIZE    = 4
GRAD_ACCUM    = 8
LEARNING_RATE = 2e-5
# =======================================================

## 3. Načítanie datasetu (Roboflow Formát)

In [ ]:
def load_roboflow_florence_dataset(jsonl_path: str, image_folder: str):
    records = []
    image_folder = Path(image_folder)
    
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] Chyba na riadku {line_num}: {e}")
                continue
            
            image_file = record.get("image") or record.get("file_name") or record.get("filename")
            # Roboflow exportuje prompt do rôznych kľúčov, zachytávame všetky možnosti
            prefix = record.get("prefix") or record.get("prompt") or record.get("question") or "<OD>"
            suffix = record.get("suffix") or record.get("completion") or record.get("answer") or ""
            
            if not image_file or not suffix: continue
            
            img_path = image_folder / image_file
            if not img_path.exists():
                continue
            
            records.append({
                "image_path": str(img_path),
                "prefix": prefix,
                "suffix": suffix
            })
            
    print(f"Načítaných {len(records)} záznamov z Roboflow datasetu.")
    return records

all_records = load_roboflow_florence_dataset(DATASET_PATH, IMAGE_FOLDER)
print("Ukážka:", {k: v for k, v in all_records[0].items() if k != 'image_path'})

import random
random.seed(42)
random.shuffle(all_records)

split_idx  = int(len(all_records) * 0.9)
train_data = all_records[:split_idx]
val_data   = all_records[split_idx:]
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

## 4. Inicializácia Modelu a Processora (obsahuje fix pre Transformers)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    device_map="auto"
)

# === HACK PRE NOVÚ VERZIU TRANSFORMERS (Záchrana pre Florence-2) ===
if hasattr(model.config, 'text_config') and not hasattr(model.config.text_config, 'forced_bos_token_id'):
    model.config.text_config.forced_bos_token_id = None
if not hasattr(model.config, 'forced_bos_token_id'):
    model.config.forced_bos_token_id = None
# ===================================================================

# LoRA Konfigurácia pre Florence-2
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Collate Funkcia pre Florence-2

In [ ]:
def collate_fn(batch):
    prompts = []
    answers = []
    images = []
    
    for ex in batch:
        try:
            img = Image.open(ex["image_path"]).convert("RGB")
        except:
            continue
        images.append(img)
        prompts.append(ex["prefix"])
        answers.append(ex["suffix"])
        
    if not images: return None

    # Zabalíme obrázky a prompty
    inputs = processor(
        text=prompts, 
        images=images, 
        return_tensors="pt", 
        padding=True
    )
    
    # Zabalíme správne odpovede (tzv. labels)
    labels = processor.tokenizer(
        text=answers, 
        return_tensors="pt", 
        padding=True, 
        return_attention_mask=False
    )["input_ids"]
    
    # Maskovanie padding tokenov, aby sa z nich model neučil (-100 v PyTorchi znamená "ignoruj")
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    
    # Presun obrazových matíc do správneho dátového typu
    inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)
    
    return inputs

print("Collate funkcia pripravená.")

## 6. Tréning

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=1e-6,
    adam_beta2=0.999,
    warmup_ratio=0.03,
    bf16=True,                       
    tf32=True,                       
    optim="adamw_torch_fused",       
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    remove_unused_columns=False,     # Nutné pre custom collate_fn
    dataloader_num_workers=0,        
    dataloader_pin_memory=False,     
    report_to=["tensorboard"],
    run_name="florence2-weapon-finetune",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=collate_fn,
)

print("Spúšťam tréning Florence-2...")
trainer.train()

## 7. Uloženie

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model uložený do zložky: {OUTPUT_DIR}")

if HF_REPO_ID:
    from huggingface_hub import login
    import os
    from dotenv import load_dotenv
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))
    
    trainer.push_to_hub(HF_REPO_ID)
    processor.push_to_hub(HF_REPO_ID)

## 8. Inferencia (Otestovanie Modelu)

In [ ]:
inf_processor = AutoProcessor.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto"
)

# Hack pre inferenciu
if hasattr(base_model.config, 'text_config') and not hasattr(base_model.config.text_config, 'forced_bos_token_id'):
    base_model.config.text_config.forced_bos_token_id = None
if not hasattr(base_model.config, 'forced_bos_token_id'):
    base_model.config.forced_bos_token_id = None

inf_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
inf_model.eval()

def predict_florence(image_path, prompt="<OD>"):
    img = Image.open(image_path).convert("RGB")
    inputs = inf_processor(text=prompt, images=img, return_tensors="pt").to(device, DTYPE)

    with torch.no_grad():
        generated_ids = inf_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=512,
            num_beams=3,
        )

    generated_text = inf_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    
    # Florence post-processing konvertuje tokeny späť na súradnice
    parsed_answer = inf_processor.post_process_generation(
        generated_text, 
        task=prompt, 
        image_size=(img.width, img.height)
    )
    return parsed_answer

# Test
if val_data:
    sample = val_data[0]
    print(f"Testujem obrázok: {sample['image_path']}")
    vysledok = predict_florence(sample['image_path'], prompt="<OD>")
    print("\nDetekcie:", vysledok)